In [ ]:
# Выполни в ядре Python 3.12. Установка относится именно к этому ядру.
import importlib.util
import subprocess
import sys
from pathlib import Path

# uv может создать окружение без pip; ensurepip использует встроенный комплект Python.
if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "modules/tasks.json").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Открой notebook из корня курса или modules/01-ml")
requirements_path = ROOT / "labs/housing/requirements.txt"
housing_package = ROOT / "labs/housing"
%pip install -r "{requirements_path}" ipykernel==6.29.5 matplotlib==3.10.3
%pip install --no-deps -e "{housing_package}"

# ML: от данных к сохранённой модели

Здесь ты обучишь модель прогнозирования медианной стоимости жилья в районе на California Housing. Одного `fit()` недостаточно: нужно отделить данные оценки, сравнить модель с простым ответом, исследовать ошибки и сохранить воспроизводимый результат.

[Теория](README.md) · [Подготовка](README.md#подготовка-к-заданиям) · [Окружение](../../SETUP.md) · [Контракт артефактов](../../labs/housing/README.md)

## Как работать

Выбери ядро существующей `.venv` с Python 3.12. Первая ячейка устанавливает зависимости в это ядро. Если после установки редактор предлагает перезапуск, перезапусти ядро и продолжи с импортов. Новое окружение создавать не нужно. Если редактор предлагает установить ipykernel в выбранную `.venv`, разреши установку, чтобы ядро смогло запуститься.

Выполняй ячейки сверху вниз. Функции с `NotImplementedError` реализуешь ты. Следующая проверочная ячейка до реализации должна падать. Измени функцию, повторно выполни её ячейку и затем проверочную: она использует текущее определение в ядре. Никаких `make task` для этого модуля нет.

Тесты работают на маленьких синтетических данных и во временных папках. Они проверяют механику, но не подтверждают качество California Housing. После перезапуска ядра все необходимые определения нужно выполнить заново. Перед использованием `housing.train` в других модулях сохрани notebook: CLI читает сохранённые определения с диска.

## Оглавление

1. [Данные](#данные)
2. [Baseline](#baseline): метрики, split, постоянный прогноз.
3. [Кандидат](#кандидат): Pipeline и воспроизводимое обучение.
4. [Анализ ошибок](#анализ-ошибок): группы и географическая граница.
5. [Сохранение результата](#сохранение-результата): четыре файла для следующих модулей.

Ячейки `course:imports` и `course:definitions` содержат обычный Python для `housing.train`/`housing.evaluate`. Не помещай в них чтение данных, магические команды, графики, проверки или вызовы обучения. Это позволяет запускать обучение из Airflow и ClearML без Jupyter. Остальные ячейки эти процессы не выполняют.


In [ ]:
import argparse
import hashlib
import json
import os
import platform
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from course_tasks import course_root, read_json
from course_tasks.artifacts import export_files
from housing.features import FEATURES, TARGET

In [ ]:
# Эти библиотеки нужны только интерактивным ячейкам, не CLI обучения.
import matplotlib.pyplot as plt
import pytest
from tempfile import TemporaryDirectory

<a id="данные"></a>

## Данные

На входе `work/housing.csv`: восемь признаков и `MedHouseVal`. Одна строка описывает район, а не дом. Значение target `2.5` означает 250 000 USD в историческом датасете, а не актуальную рыночную оценку.

| Признак | Значение |
|---|---|
| `MedInc` | Медианный доход района |
| `HouseAge` | Медианный возраст домов |
| `AveRooms`, `AveBedrms` | Среднее количество комнат и спален |
| `Population`, `AveOccup` | Население и средняя заселённость |
| `Latitude`, `Longitude` | Географические координаты |

Если CSV ещё нет, явно выполни из терминала в корне курса `python -m housing.data --output work/housing.csv`. Команда требует сеть только при первой загрузке. При существующем файле она откажется перезаписывать данные. Сохрани этот CSV для последующих модулей.

Признаки передаются в модель в порядке `FEATURES`, target в её вход не входит. Позиционные индексы split применяются через `.iloc`, а метки географического разбиения — через `.loc`.


In [ ]:
ROOT = course_root()
DATA = ROOT / "work/housing.csv"
SEED = 42
if not DATA.is_file():
    raise FileNotFoundError(
        "Сначала создай work/housing.csv: python -m housing.data --output work/housing.csv"
    )
frame = pd.read_csv(DATA)
assert set(frame.columns) == set(FEATURES + [TARGET])
frame[FEATURES + [TARGET]].head()

In [ ]:
# Посмотри типы, пропуски и диапазоны перед обучением.
frame[FEATURES + [TARGET]].describe().T

In [ ]:
ax = frame[TARGET].hist(bins=40, figsize=(8, 3))
ax.set(
    xlabel="MedHouseVal, 100000 USD",
    ylabel="Количество районов",
    title="Распределение target",
)
plt.show()

<a id="baseline"></a>

## Baseline

Допустим, сервис пока отвечает одним числом на любой район. Признаки он не использует, но его ошибку уже можно измерить. Такая простая модель станет **baseline**, точкой отсчёта: сложная модель должна быть полезнее этого ответа.

Сначала реализуй общие правила оценки и отдели строки для проверки. Затем обучи `DummyRegressor(strategy="mean")`: он запоминает среднее target только обучающей выборки и возвращает его для всех новых строк. На train `[1, 2, 9]` это `4`, а не медиана `2`.

### Метрики

Нужно выразить качество прогноза числом, чтобы сравнивать модели на одних строках. **MAE** усредняет абсолютные ошибки. **RMSE** сначала возводит ошибки в квадрат, затем берёт корень из их среднего. Поэтому большая отдельная ошибка сильнее влияет на RMSE.

В маленьком примере истинные стоимости двух районов равны `[0, 2]`, а прогнозы — `[0, 0]` в единицах target. Ошибки равны 0 и 2: MAE = 1, RMSE = √2. Нулевое значение здесь выбрано для удобства расчёта, а не как типичная стоимость района.

Твоя `regression_metrics(y_true, y_pred) -> dict[str, float]` возвращает ровно `mae` и `rmse`. Обе метрики остаются в единицах target. Подробнее: [метрики и baseline](README.md#baseline-и-метрики).

### Аргументы `regression_metrics`

| Аргумент | Тип и формат | Происхождение и пример |
|---|---|---|
| `y_true` | Непустая одномерная последовательность конечных чисел | Истинные значения target для оцениваемых строк, например `[0.0, 2.0]` |
| `y_pred` | Одномерная последовательность такой же длины | Прогнозы для тех же строк в том же порядке, например `[0.0, 0.0]` |

Результат для примера — `{"mae": 1.0, "rmse": 2 ** 0.5}`. Число строк, порядок и единицы должны совпадать. Пустой массив, другая размерность или длина, NaN и Infinity с любой стороны вызывают `ValueError`.


In [ ]:
def regression_metrics(y_true, y_pred) -> dict[str, float]:
    # TODO: проверить форму и конечность массивов, вычислить MAE и RMSE.
    raise NotImplementedError("Реализуй метрики")

### Проверка метрик

Проверки ниже различают MAE и RMSE, проверяют дробные числа, идеальный прогноз, пустой ввод, разные длины, размерности, NaN и ±Infinity с обеих сторон. Все ожидаемые значения вычислены независимо от учебных функций. Выполни всю ячейку: просто определить `test_*` недостаточно, вызовы уже добавлены внизу.


In [ ]:
def test_metrics():
    assert regression_metrics([0, 2], [0, 0]) == {
        "mae": 1.0,
        "rmse": pytest.approx(2**0.5),
    }
    assert regression_metrics([1, 1], [1, 1]) == {"mae": 0.0, "rmse": 0.0}


test_metrics()


def test_invalid_metrics(a, b):
    with pytest.raises(ValueError):
        regression_metrics(a, b)


for a, b in [
    ([], []),
    ([1], [1, 2]),
    ([float("nan")], [0]),
    ([[1]], [[1]]),
    ([0], [float("inf")]),
]:
    test_invalid_metrics(a, b)


def test_metrics_varied_values(truth, predicted):
    errors = np.asarray(truth) - np.asarray(predicted)
    assert regression_metrics(truth, predicted) == {
        "mae": pytest.approx(np.abs(errors).mean()),
        "rmse": pytest.approx(np.sqrt(np.square(errors).mean())),
    }


for truth, predicted in [
    ([1, -2, 8], [4, -2, 0]),
    ([0.5], [-1.25]),
    ([3, 5, 9, -2], [1] * 4),
]:
    test_metrics_varied_values(truth, predicted)


def test_metrics_reject_invalid_on_either_side(bad, side):
    args = [[0], [0]]
    args[side] = bad
    with pytest.raises(ValueError):
        regression_metrics(*args)


for bad in [[float("nan")], [float("inf")], [-float("inf")], [[1, 2]], [[[1]]], 1.0]:
    for side in [0, 1]:
        test_metrics_reject_invalid_on_either_side(bad, side)

print("Проверки пройдены")

### Разбиение

Если учить модель и проверять её на одних строках, она может запомнить примеры вместо закономерности. Поэтому до обучения раздели исходную таблицу: **train** используется для `fit`, **validation** — для выбора конфигурации, **test** — для окончательной оценки уже выбранного кандидата.

`make_split(n_rows: int, seed: int = 42) -> dict` возвращает списки исходных **позиций** строк для этих трёх частей с долями 60/20/20. Сохранённые позиции позволяют использовать одно и то же разбиение у baseline и кандидата, а позже пересчитать их метрики.

Используй два вызова `train_test_split` с одним seed: сначала `test_size=0.2`, затем `test_size=0.25` от оставшихся строк. Вторая доля выделяет 20% исходной таблицы, поскольку 0.25 × 0.8 = 0.2. Не округляй доли самостоятельно: тесты учитывают поведение sklearn на нечётном числе строк.

Даже подготовка признаков должна обучаться только на train. Вычислить среднее для заполнения пропусков по всей таблице — значит использовать сведения из будущей оценки. Это **утечка данных**, подробнее в [разделе о разбиении](README.md#разбиение-и-утечка).

#### Аргументы `make_split`

| Аргумент | Тип и формат | Происхождение и пример |
|---|---|---|
| `n_rows` | `int`, число строк исходной таблицы | Передаётся как `len(frame)`. Например, `100` |
| `seed` | `int`, по умолчанию `42` | Управляет перемешиванием sklearn и позволяет повторить то же разбиение |

Результат имеет ключи `train`, `validation`, `test`. Значения — списки позиций от `0` до `n_rows - 1`, а не метки DataFrame. На 100 строках длины списков равны 60, 20 и 20. Позиции не должны пересекаться или теряться. После получения разбиения используй его во всех сравниваемых моделях.


In [ ]:
def make_split(n_rows: int, seed: int = 42) -> dict:
    # TODO: двухэтапный train_test_split и исходные позиционные индексы.
    raise NotImplementedError("Разбей данные")

### Проверка split

На 100 строках ожидаются 60/20/20. Важны не только размеры: проверяем точные позиции, повторяемость и округление для разных размеров и seed.


In [ ]:
def test_split_exact_and_deterministic():
    part = make_split(100, 42)
    trainval, test = train_test_split(list(range(100)), test_size=0.2, random_state=42)
    train, valid = train_test_split(trainval, test_size=0.25, random_state=42)
    assert part == {"train": list(train), "validation": list(valid), "test": list(test)}
    assert part == make_split(100, 42)


test_split_exact_and_deterministic()


def test_split_rounding_and_seeds(n, seed):
    trainval, test = train_test_split(list(range(n)), test_size=0.2, random_state=seed)
    train, valid = train_test_split(trainval, test_size=0.25, random_state=seed)
    expected = {"train": list(train), "validation": list(valid), "test": list(test)}
    assert make_split(n, seed) == expected
    assert make_split(n, seed) == expected


for n, seed in [(5, 0), (7, 17), (11, 42), (23, 9), (101, 3)]:
    test_split_rounding_and_seeds(n, seed)

print("Проверки пройдены")

### Постоянный прогноз

Теперь есть метрики и фиксированные части таблицы. Свяжи их в `fit_baseline(frame, split: dict) -> dict`: функция обучает постоянный ответ на train и измеряет его ошибку отдельно на validation и test.

Проверь состав столбцов: ровно `FEATURES + [TARGET]`. Обучи `DummyRegressor(strategy="mean")` только на train и верни `validation`/`test` с `mae`/`rmse`. Разбиение не меняется ради улучшения метрик.

В проверочном примере train-target равен `[1, 2, 9]`, поэтому любой прогноз равен 4. На validation `[100]` MAE будет 96. На test `[20, 40]` абсолютные ошибки равны 16 и 36: MAE = 26, RMSE = √776. Нестандартные метки DataFrame не должны менять позиционное разбиение.

#### Аргументы `fit_baseline`

| Аргумент | Тип и формат | Происхождение и пример |
|---|---|---|
| `frame` | `pandas.DataFrame` со столбцами ровно `FEATURES + [TARGET]` | Прочитанный CSV или маленькая тестовая таблица. Target остаётся в таблице, но не передаётся в признаки |
| `split` | `dict` с позициями `train`, `validation`, `test` | Результат `make_split`. Например, `{"train": [0, 1, 2], "validation": [3], "test": [4, 5]}` |

Верни `{"validation": {"mae": ..., "rmse": ...}, "test": {"mae": ..., "rmse": ...}}`. Каждая метрика рассчитывается на своей выборке. Состав столбцов проверяется до обучения. Произвольные метки индекса не меняют позиции: для этого разбиения нужна позиционная адресация строк.


In [ ]:
def fit_baseline(frame, split: dict) -> dict:
    # TODO: проверить схему, обучить DummyRegressor только на train и оценить.
    raise NotImplementedError("Обучи baseline")

### Проверка baseline

Проверяем отсутствие утечки, среднее вместо медианы, обращение по позициям и отказ при отсутствующем либо лишнем столбце.


In [ ]:
def test_baseline_uses_train_only():
    f = pd.DataFrame({name: [1.0, 2.0, 3.0, 4.0, 5.0] for name in FEATURES})
    f[TARGET] = [1.0, 3.0, 100.0, 20.0, 40.0]
    p = {"train": [0, 1], "validation": [2], "test": [3, 4]}
    result = fit_baseline(f, p)
    assert result["validation"]["mae"] == 98.0
    assert result["test"]["mae"] == 28.0
    with pytest.raises(ValueError):
        fit_baseline(f.assign(unexpected=1), p)


test_baseline_uses_train_only()


def test_baseline_uses_positions_and_asymmetric_train_mean():
    frame = pd.DataFrame(
        {name: list(range(6)) for name in FEATURES}, index=[90, 10, 80, 20, 70, 30]
    )
    frame[TARGET] = [1.0, 2.0, 9.0, 100.0, 20.0, 40.0]
    split = {"train": [0, 1, 2], "validation": [3], "test": [4, 5]}
    result = fit_baseline(frame, split)
    assert result["validation"] == {"mae": 96.0, "rmse": 96.0}
    assert result["test"] == {"mae": 26.0, "rmse": pytest.approx(776**0.5)}
    for column in FEATURES + [TARGET]:
        with pytest.raises(ValueError):
            fit_baseline(frame.drop(columns=column), split)


test_baseline_uses_positions_and_asymmetric_train_mean()

print("Проверки пройдены")

In [ ]:
split = make_split(len(frame), SEED)
baseline_metrics = fit_baseline(frame, split)
pd.DataFrame(baseline_metrics).T

<a id="кандидат"></a>

## Кандидат

Постоянный ответ готов. Теперь модель должна учитывать доход, возраст домов и остальные признаки. Но сначала числа нужно подготовить: например, заполнить пропуски и привести разные масштабы к сопоставимым значениям.

Эти преобразования тоже имеют обученные настройки. Среднее для заполнения пропуска или масштаб признака нельзя заново вычислять по каждому запросу API. Сохранять и применять подготовку вместе с регрессором позволяет `Pipeline` — последовательность шагов scikit-learn. При `fit` обучаются преобразования и регрессор, при `predict` используются их сохранённые настройки.

Последний шаг, который учится связывать признаки с target, называют estimator. Если сохранить только его, сервис потеряет подготовку признаков. Для классификатора текста аналогичная цепочка могла бы сохранять превращение текста в числовые признаки вместе с классификатором. Подробный пример: [Pipeline](README.md#pipeline-и-модели).

### Построение Pipeline

`build_pipeline(seed: int = 42) -> sklearn.pipeline.Pipeline` возвращает необученный Pipeline с обработкой признаков и регрессором из scikit-learn. Импорты `SimpleImputer`, `StandardScaler`, `Ridge`, `RandomForestRegressor` подготовлены. Для случайного алгоритма зафиксируй seed.

#### Аргументы `build_pipeline`

| Аргумент | Тип и формат | Происхождение и пример |
|---|---|---|
| `seed` | `int`, по умолчанию `42` | Та же настройка случайности, которую получит обучение. Передаётся случайным компонентам модели |

Результат — ещё не обученный `Pipeline`. Вызывающий код отдельно передаст train-признаки и train-target в `fit`. Вход для последующего `predict` — DataFrame с восемью столбцами в порядке `FEATURES`. Подготовка признаков должна оставаться внутри сохраняемого Pipeline.


In [ ]:
def build_pipeline(seed: int = 42) -> Pipeline:
    # TODO: preprocessing и estimator из scikit-learn, без fit на полной таблице.
    raise NotImplementedError("Создай Pipeline")

### Проверка Pipeline

Маленькая синтетическая таблица проверяет обучение, форму прогнозов, конечность и сохранение/загрузку всего Pipeline. Она не заменяет проверку качества на housing.


In [ ]:
def test_pipeline_fits_and_roundtrips(tmp_path):
    rng = np.random.default_rng(42)
    X = pd.DataFrame(rng.normal(size=(80, 8)), columns=FEATURES)
    y = 2 * X["MedInc"] + X["Latitude"]
    p = build_pipeline(42)
    assert isinstance(p, Pipeline)
    p.fit(X.iloc[:60], y.iloc[:60])
    pred = p.predict(X.iloc[60:])
    assert pred.shape == (20,) and np.isfinite(pred).all()
    joblib.dump(p, tmp_path / "p.joblib")
    assert np.allclose(pred, joblib.load(tmp_path / "p.joblib").predict(X.iloc[60:]))


with TemporaryDirectory() as directory:
    tmp_path = Path(directory)
    test_pipeline_fits_and_roundtrips(tmp_path)

print("Проверки пройдены")

In [ ]:
rng = np.random.default_rng(42)
x = pd.DataFrame(rng.normal(size=(40, len(FEATURES))), columns=FEATURES)
y = 2 * x["MedInc"] + 1
pipeline_for_check = build_pipeline(seed=42)
pipeline_for_check.fit(x, y)
predictions_for_check = pipeline_for_check.predict(x.iloc[:3])
assert predictions_for_check.shape == (3,)
assert np.isfinite(predictions_for_check).all()
assert regression_metrics(np.array([1.0, 3.0]), np.array([1.0, 1.0])) == {
    "mae": 1.0,
    "rmse": pytest.approx(2**0.5),
}
print("Проверки формы и единиц пройдены")

### Программа обучения

Отдельные функции уже умеют делить строки, оценивать постоянный ответ и собирать модель. Теперь нужно запускать этот процесс целиком по пути к CSV, не повторяя действия вручную в редакторе.

`train(data: Path, output: Path, seed: int, *, parameters: dict | None = None) -> None` читает данные, обучает и выбирает модель, затем сохраняет результат одного запуска. Позже Airflow и ClearML будут вызывать эту функцию без интерактивного ядра. Поэтому она не должна зависеть от глобальных `frame`, `DATA`, `split` или результатов ранее выполненных ячеек.

Требуется:

1. Получить split через `make_split`. Посчитать baseline через `fit_baseline`.
2. На одном split сравнить Dummy, Ridge и небольшой RandomForest. Preprocessing обучать только на train, конфигурацию выбирать по validation.
3. Зафиксировать выбор и один раз оценить выбранного кандидата на test.
4. В `output` сохранить четыре файла из таблицы ниже. Полную модель сериализовать с preprocessing. Модель заменять атомарно, при ошибке сериализации сохранять прежний активный файл.

| Файл | Контракт |
|---|---|
| `model.joblib` | Bundle: `schema_version=1`, обученный `pipeline`, `features=FEATURES`, непустой `model_version` |
| `metrics.json` | `baseline` и `candidate`, у каждого `validation` и `test`, внутри `mae` и `rmse` |
| `manifest.json` | `seed`, `data_sha256`, `versions`, `parameters`, `feature_order`, `split` |
| `split.json` | Исходные позиционные индексы `train`, `validation`, `test` |

SHA-256 относится к исходным bytes CSV, `manifest.split` совпадает с `split.json`. Метрики должны соответствовать сохранённому Pipeline и тем же строкам. На полном housing ожидается `candidate.test.mae <= 0.9 * baseline.test.mae`. Если порог не достигнут, не используй test для повторного подбора. Исследование и выбор конфигурации проводи на train/validation.

Одинаковые данные и seed дают одинаковый split и близкие прогнозы. Версии библиотек также являются частью воспроизводимости. [Подробный контракт](../../labs/housing/README.md).

#### Аргументы `train` и командная строка

| Аргумент функции | Тип и формат | Откуда берётся | Пример |
|---|---|---|---|
| `data` | `pathlib.Path` к существующему CSV | Путь к тем данным, из которых получаются признаки и target | `Path("work/housing.csv")` |
| `output` | `pathlib.Path` к папке результата | Выбранное место для четырёх согласованных артефактов отдельного запуска | `Path("work/runs/example")` |
| `seed` | `int` | Общая настройка для разбиения и случайного алгоритма | `42` |
| `parameters` | Именованный `dict` или `None`, по умолчанию `None` | Расширение для [модуля 04](../04-mlflow/tasks/01-experiments/README.md). При словаре обучается один Ridge с переданными настройками | `{"alpha": 1.0}` |

Функция возвращает `None`. Результат её работы находится в файлах внутри `output`. Относительные пути отсчитываются от текущей рабочей директории CLI. В notebook подготовленные пути задают ячейки запуска.

Готовая `main()` не принимает Python-аргументов. Она разбирает обязательные `--data` и `--output`, а также `--seed` с умолчанием 42, затем вызывает твою `train`. Например, `python -m housing.train --data work/housing.csv --output work/runs/example --seed 42`. Реализовывать второй сценарий обучения для CLI не требуется.

На этапе модуля 01 достаточно режима `parameters=None`. В модуле 04 нужно доработать эту же функцию: при словаре она обучает одну заданную конфигурацию Ridge, сохраняет baseline и тот же комплект файлов. Не выбирай другую конфигурацию внутри этого режима. Набор для проверки качества выбранной модели и правило публикации остаются прежними. В манифесте сохраняются настройки фактически обученного estimator. MLflow внутри `train` не вызывается.


In [ ]:
def train(
    data: Path,
    output: Path,
    seed: int,
    *,
    parameters: dict | None = None,
) -> None:
    # TODO: сравнение кандидатов, выбор по validation, финальный test и четыре артефакта.
    # parameters=None: сценарий модуля 01. Переданный dict: один Ridge для модуля 04.
    # Настройки передаются в Ridge, а фактические значения сохраняются в manifest.parameters.
    # Используй функции выше; все необходимые входы переданы аргументами.
    raise NotImplementedError("Реализуй обучение")


def main() -> None:
    # Готовый вход для python -m housing.train; при импорте не вызывается.
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    train(args.data, args.output, args.seed)

### Проверка артефактов и повторного обучения

Проверяются четыре согласованных файла, независимый пересчёт метрик сохранённой модели на её split, SHA-256 и повторяемость. Затем искусственная ошибка сериализации должна оставить предыдущую модель читаемой и неизменной. Временные папки и подмена сериализатора очищаются даже при падении проверки.


In [ ]:
def test_training_artifacts_and_repeat(tmp_path):
    rng = np.random.default_rng(7)
    frame = pd.DataFrame(rng.uniform(1, 3, (100, 8)), columns=FEATURES)
    frame[TARGET] = 2 * frame.MedInc + frame.Latitude
    path = tmp_path / "input.csv"
    frame.to_csv(path, index=False)
    outputs = [tmp_path / "one", tmp_path / "two"]
    predictions = []
    for out in outputs:
        train(path, out, 42)
        bundle = joblib.load(out / "model.joblib")
        assert bundle["schema_version"] == 1 and bundle["features"] == FEATURES
        assert isinstance(bundle["model_version"], str) and bundle["model_version"]
        predictions.append(bundle["pipeline"].predict(frame[FEATURES].iloc[:20]))
        metrics = read_json(out / "metrics.json")
        for model in ["baseline", "candidate"]:
            for part in ["validation", "test"]:
                assert set(metrics[model][part]) == {"mae", "rmse"}
                assert all(
                    (np.isfinite(v) and v >= 0 for v in metrics[model][part].values())
                )
        manifest = read_json(out / "manifest.json")
        assert manifest["data_sha256"] == hashlib.sha256(path.read_bytes()).hexdigest()
        assert manifest["seed"] == 42 and manifest["feature_order"] == FEATURES
        assert manifest["versions"] and manifest["parameters"]
        split = read_json(out / "split.json")
        assert manifest["split"] == split
        assert sorted(sum(split.values(), [])) == list(range(100))
        for part in ["validation", "test"]:
            positions = split[part]
            truth = frame[TARGET].iloc[positions].to_numpy()
            predictions_for_part = bundle["pipeline"].predict(
                frame[FEATURES].iloc[positions]
            )
            baseline_mean = frame[TARGET].iloc[split["train"]].mean()
            for model, predicted in [
                ("candidate", predictions_for_part),
                ("baseline", np.full(len(positions), baseline_mean)),
            ]:
                errors = truth - predicted
                assert metrics[model][part] == {
                    "mae": pytest.approx(np.abs(errors).mean()),
                    "rmse": pytest.approx(np.sqrt(np.square(errors).mean())),
                }
        assert all(
            (
                (out / name).stat().st_size > 0
                for name in [
                    "model.joblib",
                    "metrics.json",
                    "manifest.json",
                    "split.json",
                ]
            )
        )
    assert read_json(outputs[0] / "split.json") == read_json(outputs[1] / "split.json")
    assert np.allclose(*predictions, rtol=1e-06, atol=1e-08)


with TemporaryDirectory() as directory:
    tmp_path = Path(directory)
    test_training_artifacts_and_repeat(tmp_path)


def test_failed_model_serialization_preserves_previous_model(tmp_path, monkeypatch):
    rng = np.random.default_rng(71)
    frame = pd.DataFrame(rng.normal(size=(40, len(FEATURES))), columns=FEATURES)
    frame[TARGET] = frame.MedInc * 2 + frame.Latitude
    data = tmp_path / "input.csv"
    frame.to_csv(data, index=False)
    output = tmp_path / "active"
    train(data, output, 42)
    model = output / "model.joblib"
    previous_bytes = model.read_bytes()
    previous_predictions = joblib.load(model)["pipeline"].predict(frame[FEATURES])
    serialization_attempts = []

    def failing_state(self):
        serialization_attempts.append(self)
        raise OSError("simulated serialization failure")

    monkeypatch.setattr(Pipeline, "__getstate__", failing_state)
    with pytest.raises(Exception) as failure:
        train(data, output, 7)
    assert serialization_attempts
    assert not isinstance(failure.value, NotImplementedError)
    assert model.read_bytes() == previous_bytes
    assert np.allclose(
        joblib.load(model)["pipeline"].predict(frame[FEATURES]), previous_predictions
    )


with TemporaryDirectory() as directory:
    tmp_path = Path(directory)
    with pytest.MonkeyPatch.context() as monkeypatch:
        test_failed_model_serialization_preserves_previous_model(tmp_path, monkeypatch)

print("Проверки пройдены")

### Проверка конфигурации обучения для модуля 04

Эта проверка нужна после доработки `train` в [задании MLflow](../04-mlflow/tasks/01-experiments/README.md). При первом прохождении модуля 01 оставь флаг `RUN_MLFLOW_CONFIGURATION_CHECKS = False`.

В модуле 04 установи его в `True` и выполни ячейку. Она вызывает текущую `train` для трёх значений `alpha`, читает сохранённый Pipeline и проверяет, что настройки действительно применены, метрики вычислены на тех же строках, а разбиение не меняется. Чтение notebook с помощью `housing.train` эту ячейку не выполняет.


In [ ]:
RUN_MLFLOW_CONFIGURATION_CHECKS = False


def test_configured_training(tmp_path):
    rng = np.random.default_rng(47)
    frame = pd.DataFrame(rng.uniform(1, 3, (100, len(FEATURES))), columns=FEATURES)
    frame[TARGET] = 1.7 * frame.MedInc + frame.Latitude
    data = tmp_path / "input.csv"
    frame.to_csv(data, index=False)
    splits = []
    for alpha in (0.1, 1.0, 10.0):
        output = tmp_path / str(alpha)
        train(data, output, 42, parameters={"alpha": alpha})
        bundle = joblib.load(output / "model.joblib")
        pipeline = bundle["pipeline"]
        assert isinstance(pipeline, Pipeline)
        estimator = pipeline.steps[-1][1]
        assert isinstance(estimator, Ridge)
        assert estimator.alpha == pytest.approx(alpha)
        manifest = read_json(output / "manifest.json")
        split = read_json(output / "split.json")
        metrics = read_json(output / "metrics.json")
        assert manifest["parameters"]["alpha"] == pytest.approx(alpha)
        assert manifest["seed"] == 42
        assert manifest["split"] == split
        assert manifest["data_sha256"] == hashlib.sha256(data.read_bytes()).hexdigest()
        splits.append(split)
        mean = frame.iloc[split["train"]][TARGET].mean()
        for part in ("validation", "test"):
            rows = frame.iloc[split[part]]
            truth = rows[TARGET].to_numpy()
            for name, predicted in (
                ("candidate", pipeline.predict(rows[FEATURES])),
                ("baseline", np.full(len(rows), mean)),
            ):
                errors = truth - predicted
                assert metrics[name][part] == {
                    "mae": pytest.approx(np.mean(np.abs(errors))),
                    "rmse": pytest.approx(np.sqrt(np.mean(errors ** 2))),
                }
    assert splits[0] == splits[1] == splits[2]


if RUN_MLFLOW_CONFIGURATION_CHECKS:
    with TemporaryDirectory() as directory:
        test_configured_training(Path(directory))
    print("Проверки конфигураций пройдены")
else:
    print("Проверка конфигураций не включена: запусти её при прохождении модуля 04")


### Полный эксперимент

Запусти эту ячейку после прохождения маленьких проверок. Обучение временно записывает комплект в изолированную папку. Он загружается в память, папка удаляется. Рабочая модель в `work/artifacts/` здесь не меняется: публикация будет последним действием notebook.

После этого конфигурация зафиксирована. Не подбирай её по увиденному test. Произвольные числа из примеров курса не заменяют собственные метрики.


In [ ]:
with TemporaryDirectory() as directory:
    run_dir = Path(directory)
    train(DATA, run_dir, SEED)
    bundle = joblib.load(run_dir / "model.joblib")
    metrics = read_json(run_dir / "metrics.json")
    manifest = read_json(run_dir / "manifest.json")
    split = read_json(run_dir / "split.json")

comparison = pd.DataFrame(
    {
        name: {
            f"{part}_{metric}": value
            for part, values in result.items()
            for metric, value in values.items()
        }
        for name, result in metrics.items()
    }
).T
comparison

In [ ]:
assert manifest["split"] == split
assert manifest["feature_order"] == bundle["features"] == FEATURES
assert metrics["candidate"]["test"]["mae"] <= 0.9 * metrics["baseline"]["test"]["mae"]
comparison[["validation_mae", "test_mae"]].plot.bar(
    figsize=(8, 3), ylabel="MAE, 100000 USD", rot=0
)
plt.show()

<a id="анализ-ошибок"></a>

## Анализ ошибок

Общая MAE скрывает неоднородность: дорогие районы могут иметь гораздо большую ошибку. Извлеки test-строки по сохранённым позициям, затем сравни прогнозы с target тех же строк. Не соединяй независимо отсортированные массивы по позиции.

### Ошибки по квартилям

Чтобы увидеть разницу между дешёвыми и дорогими районами, раздели оцениваемые строки на четыре группы по настоящей стоимости. Такие группы по четвертям распределения называют **квартильными**. Прогнозы не определяют принадлежность к группе: иначе ошибка могла бы сама менять состав сравнения.

В `quartile_errors(y_true, y_pred) -> list[dict]` используй `pd.qcut(y_true, 4, labels=False, duplicates="drop")`. Эта операция ищет границы по target и при совпадающих границах уменьшает число групп. Результат функции — упорядоченные записи `quartile`, `count`, `mae`, `rmse`. Непустые группы нумеруются с 1. Если все targets одинаковы, отдельно верни одну группу `quartile=1`.

В условной проверке targets восьми районов равны `[0,1,2,3,4,5,6,7]`, а модель всегда отвечает 0. По два района попадают в каждую группу. MAE по порядку равны 0.5, 2.5, 4.5 и 6.5: ошибка растёт вместе со стоимостью, хотя общая метрика показывает только одно число. Подробнее: [анализ групп](README.md#анализ-групп-и-неверные-успехи).

#### Аргументы `quartile_errors`

| Аргумент | Тип и формат | Происхождение и пример |
|---|---|---|
| `y_true` | Непустая одномерная последовательность конечных чисел | Target одних и тех же test-строк, например `[0, 1, 2, 3, 4, 5, 6, 7]` |
| `y_pred` | Одномерная последовательность такой же длины | Прогнозы выбранной модели на этих строках в том же порядке. Для примера — восемь нулей |

Верни список словарей вида `{"quartile": 1, "count": 2, "mae": 0.5, "rmse": ...}`. Номер группы и её размер описывают именно строки с соответствующими target. Не дели прогнозы отдельно и не подменяй RMSE средним абсолютным отклонением. При ошибочном формате массивов ожидается `ValueError`.


In [ ]:
def quartile_errors(y_true, y_pred) -> list[dict]:
    # TODO: проверить массивы, сгруппировать по qcut и посчитать метрики каждой группы.
    raise NotImplementedError("Проанализируй ошибки")

### Проверка групп

Проверки охватывают MAE/RMSE всех групп, одинаковые и повторяющиеся targets, неравномерные группы и некорректные массивы.


In [ ]:
def test_quartiles_and_counts():
    rows = quartile_errors(list(range(8)), [0] * 8)
    assert [r["quartile"] for r in rows] == [1, 2, 3, 4]
    assert [r["count"] for r in rows] == [2] * 4
    assert [r["mae"] for r in rows] == [0.5, 2.5, 4.5, 6.5]
    assert [r["rmse"] for r in rows] == pytest.approx(
        [0.5**0.5, 6.5**0.5, 20.5**0.5, 42.5**0.5]
    )
    assert quartile_errors([2, 2], [1, 3]) == [
        {"quartile": 1, "count": 2, "mae": 1.0, "rmse": 1.0}
    ]


test_quartiles_and_counts()


def test_quartile_metrics_uneven_groups_and_ties(truth, predicted, groups):
    result = quartile_errors(truth, predicted)
    assert len(result) == len(groups)
    for quartile, (row, positions) in enumerate(zip(result, groups), start=1):
        errors = np.asarray(truth)[positions] - np.asarray(predicted)[positions]
        assert row == {
            "quartile": quartile,
            "count": len(positions),
            "mae": pytest.approx(np.abs(errors).mean()),
            "rmse": pytest.approx(np.sqrt(np.square(errors).mean())),
        }


for truth, predicted, groups in [
    ([0, 1, 2, 3, 4, 5, 6], [2, 1, 0, 4, 8, 3, 9], [[0, 1], [2, 3], [4], [5, 6]]),
    ([0, 0, 0, 1, 2, 3, 3, 3], [1, 2, 3, 1, 0, 4, 5, 6], [[0, 1, 2, 3], [4, 5, 6, 7]]),
]:
    test_quartile_metrics_uneven_groups_and_ties(truth, predicted, groups)


def test_quartiles_reject_invalid_arrays(truth, predicted):
    with pytest.raises(ValueError):
        quartile_errors(truth, predicted)


for truth, predicted in [
    ([], []),
    ([[1]], [[1]]),
    ([float("inf")], [1]),
    ([1], [float("nan")]),
    ([1, 2], [1]),
]:
    test_quartiles_reject_invalid_arrays(truth, predicted)

print("Проверки пройдены")

In [ ]:
test_rows = frame.iloc[split["test"]]
y_true = test_rows[TARGET].to_numpy()
y_pred = bundle["pipeline"].predict(test_rows[FEATURES])
errors_by_quartile = pd.DataFrame(quartile_errors(y_true, y_pred))
errors_by_quartile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].scatter(y_true, y_pred, s=4, alpha=0.25)
axes[0].set(
    xlabel="Target, 100000 USD",
    ylabel="Прогноз, 100000 USD",
    title="Прогноз против ответа",
)
axes[1].hist(y_true - y_pred, bins=40)
axes[1].set(xlabel="Ответ − прогноз, 100000 USD", ylabel="Количество", title="Остатки")
plt.tight_layout()
plt.show()

### Географическое разбиение

Случайное перемешивание не показывает перенос в новый регион. `geographic_split(frame, latitude_cutoff: float) -> dict` делит **исходные метки индекса** на `train` (`Latitude < cutoff`) и `test` (`>= cutoff`), сохраняя порядок. Пустая сторона вызывает `ValueError`. Для широт `[33,35,36]`, индекса `[10,20,30]` и границы 35 ожидаются train `[10]`, test `[20,30]`.

Это отдельный опыт: географическая модель не заменяет уже выбранный рабочий bundle. Для подбора внутри северной/южной обучающей части выдели собственную validation, не используй географический test для выбора.

#### Аргументы `geographic_split`

| Аргумент | Тип и формат | Происхождение и пример |
|---|---|---|
| `frame` | `pandas.DataFrame` со столбцом `Latitude` | Исходная таблица с её настоящими метками индекса. В примере индексы `[10, 20, 30]` и широты `[33, 35, 36]` |
| `latitude_cutoff` | `float` | Выбранная граница широты. В примере `35.0` |

Результат примера — `{"train": [10], "test": [20, 30]}`. В этой функции возвращаются метки индекса, поэтому извлекать строки нужно через `.loc`. В `make_split` использовались позиции и `.iloc`. Различие является частью контракта. Если хотя бы одна сторона пуста, функция вызывает `ValueError`.


In [ ]:
def geographic_split(frame, latitude_cutoff: float) -> dict:
    # TODO: вернуть исходные метки обеих сторон, проверить непустоту.
    raise NotImplementedError("Раздели регионы")

### Проверка географической границы

Равенство границе относится к test. Проверяются произвольные метки, порядок и обе пустые стороны. Комбинированный тест также сохраняет проверку несовпадения длин для квартилей.


In [ ]:
def test_geographic_boundary_and_validation():
    f = pd.DataFrame({"Latitude": [33.0, 35.0, 36.0]}, index=[10, 20, 30])
    assert geographic_split(f, 35.0) == {"train": [10], "test": [20, 30]}
    with pytest.raises(ValueError):
        geographic_split(f, 40.0)
    with pytest.raises(ValueError):
        quartile_errors([1, 2], [1])


test_geographic_boundary_and_validation()


def test_geographic_order_and_both_empty_sides():
    frame = pd.DataFrame(
        {"Latitude": [35.0, 34.9, 35.1, 30.0]}, index=["d", "a", "c", "b"]
    )
    assert geographic_split(frame, 35.0) == {"train": ["a", "b"], "test": ["d", "c"]}
    for cutoff in [30.0, 36.0]:
        with pytest.raises(ValueError):
            geographic_split(frame, cutoff)


test_geographic_order_and_both_empty_sides()

print("Проверки пройдены")

### Самостоятельные опыты

1. Обучи отдельный Pipeline на географическом train, оцени на географическом test и сопоставь результат со случайным разбиением. Применяй метки через `.loc`. Не перезаписывай `bundle`, `metrics`, `manifest`, `split` основного запуска.
2. На отдельной копии эксперимента намеренно выполни scaler-fit до split и сравни метрики с корректным Pipeline. Даже небольшая разница не оправдывает утечку: нарушение определяется доступом к данным, а не размером улучшения. Этот артефакт не сохраняй как рабочий.

Здесь нужны вычисления и графики, письменный отчёт не требуется. Готовых реализаций этих опытов нет.


In [ ]:
# TODO: отдельный географический опыт; используй собственные переменные geo_*.
# TODO: отдельный ошибочный опыт со scaler до split; используй переменные leaked_*.
# Основной bundle после этих опытов остаётся неизменным.


<a id="сохранение-результата"></a>

## Сохранение результата

Следующим модулям понадобится файл модели, который можно загрузить без живого ядра notebook. Вместе с ним нужны измеренные метрики и сведения о данных и разбиении. Последняя ячейка уже подготовлена: она **не обучает** модель, а сохраняет находящиеся в памяти `bundle`, `metrics`, `manifest`, `split` непосредственно в `work/artifacts/`.

Сначала весь комплект сериализуется во временную папку. Общий помощник курса `export_files` проверяет наличие файлов и конфликты, затем заменяет каждый файл атомарно. Модель переносится последней, после её сопроводительных файлов. Весь комплект при этом не является единой транзакцией: при сбое между заменами часть файлов уже может обновиться.

При существующем отличающемся результате функция откажется заменять его. После осознанного выбора используй `replace=True` в последнем вызове. Посторонние файлы не удаляются. Если API уже работает, после замены модели его нужно перезапустить.

Результат — `model.joblib`, `metrics.json`, `manifest.json`, `split.json`. Каталог `work/` исключён из Git, а notebook с твоим кодом можно сохранить в репозитории. Перед коммитом очисти результаты выполнения и проверь отсутствие личных данных. Следующие модули читают эти файлы. Повторное обучение запускается через `python -m housing.train --data work/housing.csv --output work/runs/example --seed 42` и использует сохранённые определения этого notebook.

### Аргументы готовой `save_artifacts`

| Аргумент | Тип и формат | Происхождение и назначение |
|---|---|---|
| `bundle` | `dict` с Pipeline и метаданными | Объект выбранной и обученной модели из ячейки полного эксперимента |
| `metrics` | `dict` с baseline/candidate и validation/test | Реальные MAE/RMSE этого запуска |
| `manifest` | `dict` с происхождением обучения | SHA-256 CSV, seed, версии, параметры, признаки и разбиение |
| `split` | `dict` со списками позиций | То самое разбиение, по которому модель обучалась и оценивалась |
| `replace` | Именованный `bool`, по умолчанию `False` | Разрешает замену уже существующих отличающихся файлов только при `True` |

Функция возвращает список перенесённых путей. При совпадающих файлах список пуст. При конфликте без разрешения замены возникает `FileExistsError`. Печатные пути показывают место сохранения, но не заменяют проверку модели и метрик. Реализация этой функции уже предоставлена.


In [ ]:
def save_artifacts(bundle, metrics, manifest, split, *, replace=False):
    """Сохранить подготовленный комплект; обучение и тесты не запускаются."""
    destination = course_root() / "work/artifacts"
    with TemporaryDirectory() as directory:
        staging = Path(directory)
        for name, value in [
            ("metrics.json", metrics),
            ("manifest.json", manifest),
            ("split.json", split),
        ]:
            (staging / name).write_text(
                json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                encoding="utf-8",
            )
        joblib.dump(bundle, staging / "model.joblib")
        names = ["metrics.json", "manifest.json", "split.json", "model.joblib"]
        try:
            changed = export_files(
                [(staging / name, destination / name) for name in names],
                replace=replace,
            )
        except FileExistsError as error:
            raise FileExistsError(
                str(error).replace("--replace", "replace=True")
            ) from error
    for name in names:
        print(destination / name)
    print("Сохранено" if changed else "Файлы уже совпадают")
    return changed


save_artifacts(bundle, metrics, manifest, split)